In [91]:
import gurobipy as gp
from gurobipy import GRB

In [92]:
# Nodes
N = [1,2,3,4,5,6]

# Edges (from_node, to_node)
E = [(1,2),(1,3),(2,4),(2,5),
    (3,4),(4,5),(4,6),(5,6)]

# Edge parameters
capacity = {(1,2):80000,
    (1,3):40000,
    (2,4):50000,
    (2,5):20000,
    (3,4):90000,
    (4,5):20000,
    (4,6):50000,
    (5,6):60000}

distance = {(1,2):150,
    (1,3):200,
    (2,4):95,
    (2,5):100,
    (3,4):120,
    (4,5):50,
    (4,6):75,
    (5,6):60}

transport_rate = {(1,2):0.10,
    (1,3):0.15,
    (2,4):0.24,
    (2,5):0.12,
    (3,4):0.12,
    (4,5):0.25,
    (4,6):0.08,
    (5,6):0.11}

# Convert loss per 100km to fraction per edge
loss = {(1,2):0.03,
    (1,3):0.035,
    (2,4):0.032,
    (2,5):0.015,
    (3,4):0.02,
    (4,5):0.022,
    (4,6):0.018,
    (5,6):0.031}

# Supply cost and max supply
supply_cost = {1:0.80,
    2:1.05,
    3:0.90,
    4:1.10,
    5:1.12,
    6:1.15}

max_supply = {1:100000,
    2:60000,
    3:25000,
    4:0,
    5:0,
    6:0}

# Demand at nodes
demand = {1:0,
    2:40000,
    3:30000,
    4:20000,
    5:25000,
    6:50000}


In [93]:
m = gp.Model("Olive_LMP")

# Variables
Supply = m.addVars(N, lb=0, name="Supply")
Flow = m.addVars(E, lb=0, name="Flow")

In [94]:
# Objective
supply_cost_term = gp.quicksum(
    supply_cost[n]*Supply[n] for n in N)

transport_cost_term = gp.quicksum(
    (distance[e]/100)*transport_rate[e]*Flow[e] for e in E)

m.setObjective(supply_cost_term + transport_cost_term, GRB.MINIMIZE)


In [95]:
# Constraints

# Supply limit
supply_limit = m.addConstrs(
    (Supply[n] <= max_supply[n] for n in N), name="SupplyLimit")

# Flow capacity
capacity_con = m.addConstrs(
    (Flow[e] <= capacity[e] for e in E), name="Capacity")

# Node balance constraints
balance = {}

for n in N:

    inflow = gp.quicksum(
        Flow[i,j]*(1-loss[(i,j)]*distance[i,j]/100) for (i,j) in E if j==n)
    
    outflow = gp.quicksum(
        Flow[i,j] for (i,j) in E if i==n)

    balance[n] = m.addConstr(
        Supply[n] + inflow - outflow == demand[n],
        name=f"Balance_{n}")


In [96]:
m.optimize()

print("\nOptimal Objective Value")
print(m.objVal)

print("\nOptimal Supply")
for n in N:
    print(f"Node {n}:", Supply[n].X)

print("\nOptimal Flows")
for e in E:
    print(f"{e}:", Flow[e].X)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 20 rows, 14 columns and 36 nonzeros
Model fingerprint: 0x53a27de9
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 1e+05]
Presolve removed 16 rows and 6 columns
Presolve time: 0.01s
Presolved: 4 rows, 8 columns, 14 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.6213341e+04   2.101479e+04   0.000000e+00      0s
       6    1.9561249e+05   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.956124942e+05

Optimal Objective Value
195612.4942206426

Optimal Supply
Node 1: 100000.0
Node 2: 48646.33311988724
Node 3: 25000.0
Node 4: 0.0
Node 5: 0.0
No

In [97]:
# Supply cost
supply_cost_value = sum(
    supply_cost[n] * Supply[n].X for n in N
)

# Transport cost
transport_cost_value = sum(
    (distance[e] / 100) * transport_rate[e] * Flow[e].X
    for e in E
)

# Total (sanity check)
total_cost = supply_cost_value + transport_cost_value

print("\n--- Cost Breakdown ---")
print(f"Supply Cost:        {supply_cost_value:,.2f}")
print(f"Transport Cost:     {transport_cost_value:,.2f}")
print(f"Total Cost:         {total_cost:,.2f}")
print(f"Gurobi Objective:   {m.objVal:,.2f}")


--- Cost Breakdown ---
Supply Cost:        153,578.65
Transport Cost:     42,033.84
Total Cost:         195,612.49
Gurobi Objective:   195,612.49


In [98]:
#Dual Values
print("\nDual Values (Shadow Prices)")

print("\nNode Balance Duals")
for n in N:
    print(f"Node {n}:", balance[n].Pi)
    
# print("\nSupply Limit Duals")
# for n in N:
#     print(f"Node {n}:", supply_limit[n].Pi)

# print("\nCapacity Duals")
# for e in E:
#     print(f"{e}:", capacity_con[e].Pi)


Dual Values (Shadow Prices)

Node Balance Duals
Node 1: 0.85275
Node 2: 1.05
Node 3: 1.2395161290322583
Node 4: 1.4175370174510842
Node 5: 1.5596936475744028
Node 6: 1.656504633762383


## Scenario 1: Double transport capacity on constrained links

In [99]:
#Scenario 1: Double transport capacity on constrained links

# Edge parameters
capacity1 = {(1,2):80000,
    (1,3):40000,
    (2,4):100000,
    (2,5):40000,
    (3,4):90000,
    (4,5):20000,
    (4,6):100000,
    (5,6):60000}

m1 = gp.Model("Olive_LMP_Sc1")

# Variables
Supply = m1.addVars(N, lb=0, name="Supply")
Flow = m1.addVars(E, lb=0, name="Flow")

# Objective
supply_cost_term = gp.quicksum(
    supply_cost[n]*Supply[n] for n in N)

transport_cost_term = gp.quicksum(
    distance[e]/100*transport_rate[e]*Flow[e] for e in E)

m1.setObjective(supply_cost_term + transport_cost_term, GRB.MINIMIZE)

# Constraints

# Supply limit
supply_limit = m1.addConstrs(
    (Supply[n] <= max_supply[n] for n in N), name="SupplyLimit")

# Flow capacity
capacity_con = m1.addConstrs(
    (Flow[e] <= capacity1[e] for e in E), name="Capacity")

# Node balance constraints
balance = {}

for n in N:

    inflow = gp.quicksum(
        Flow[i,j]*(1-loss[(i,j)]*distance[i,j]/100) for (i,j) in E if j==n)

    outflow = gp.quicksum(
        Flow[i,j] for (i,j) in E if i==n)

    balance[n] = m1.addConstr(
        Supply[n] + inflow - outflow == demand[n],
        name=f"Balance_{n}")

m1.optimize()

print("\nOptimal Objective Value")
print(m1.objVal)

print("\nOptimal Supply")
for n in N:
    print(f"Node {n}:", Supply[n].X)

print("\nOptimal Flows")
for e in E:
    print(f"{e}:", Flow[e].X)
    
#Dual Values
print("\nDual Values (Shadow Prices)")

print("\nNode Balance Duals")
for n in N:
    print(f"Node {n}:", balance[n].Pi)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 20 rows, 14 columns and 36 nonzeros
Model fingerprint: 0x17206020
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 1e+05]
Presolve removed 17 rows and 7 columns
Presolve time: 0.01s
Presolved: 3 rows, 7 columns, 12 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8010731e+04   2.033220e+04   0.000000e+00      0s
       3    1.9047974e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.904797414e+05

Optimal Objective Value
190479.7414018708

Optimal Supply
Node 1: 87219.77151620916
Node 2: 60000.0
Node 3: 25000.0
Node 4: 0.0
Node 5: 0.0
Nod

## Scenario 2: Meet all demand from local suppliers

In [100]:
#Scenario 2: Meet all demand from local suppliers

# Supply cost and adjusted max supply
supply_cost = {1:0.80,
    2:1.05,
    3:0.90,
    4:1.10,
    5:1.12,
    6:1.15}

max_supply2 = {1:0,
    2:40000,
    3:30000,
    4:20000,
    5:25000,
    6:50000}

m2 = gp.Model("Olive_LMP_Sc2")

# Variables
Supply = m2.addVars(N, lb=0, name="Supply")
Flow = m2.addVars(E, lb=0, name="Flow")

# Objective
supply_cost_term = gp.quicksum(
    supply_cost[n]*Supply[n] for n in N)

transport_cost_term = gp.quicksum(
    distance[e]/100*transport_rate[e]*Flow[e] for e in E)

m2.setObjective(supply_cost_term + transport_cost_term, GRB.MINIMIZE)

# Constraints

# Supply limit
supply_limit = m2.addConstrs(
    (Supply[n] <= max_supply2[n] for n in N), name="SupplyLimit")

# Flow capacity
capacity_con = m2.addConstrs(
    (Flow[e] <= capacity[e] for e in E), name="Capacity")

# Node balance constraints
balance = {}

for n in N:

    inflow = gp.quicksum(
        Flow[i,j]*(1-loss[(i,j)]*distance[i,j]/100) for (i,j) in E if j==n)

    outflow = gp.quicksum(
        Flow[i,j] for (i,j) in E if i==n)

    balance[n] = m2.addConstr(
        Supply[n] + inflow - outflow == demand[n],
        name=f"Balance_{n}")

m2.optimize()

print("\nOptimal Objective Value")
print(m2.objVal)

print("\nOptimal Supply")
for n in N:
    print(f"Node {n}:", Supply[n].X)

print("\nOptimal Flows")
for e in E:
    print(f"{e}:", Flow[e].X)
    
#Dual Values
print("\nDual Values (Shadow Prices)")

print("\nNode Balance Duals")
for n in N:
    print(f"Node {n}:", balance[n].Pi)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 20 rows, 14 columns and 36 nonzeros
Model fingerprint: 0xf5969f6d
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 9e+04]
Presolve removed 20 rows and 14 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.7650000e+05   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.765000000e+05

Optimal Objective Value
176500.0

Optimal Supply
Node 1: 0.0
Node 2: 40000.0
Node 3: 30000.0
Node 4: 20000.0
Node 5: 25000.0
Node 6: 50000.0

Optimal Flows
(1, 2): 0.0
(1, 3): 0.0
(2, 4): 0.0
(2, 5): 0.0
(3, 4)

## Scenario 3: Increase supply at Node 6

In [101]:
#Scenario 3: Increase supply at Node 6

# Edges (from_node, to_node)
E3 = [(1,2),(1,3),(2,4),(2,5),
    (3,4),(4,5),(4,6),(5,6),
    (6,4),(6,5)]

# Edge parameters
capacity3 = {(1,2):80000,
    (1,3):40000,
    (2,4):50000,
    (2,5):20000,
    (3,4):90000,
    (4,5):20000,
    (4,6):50000,
    (5,6):60000,
    (6,4):50000,
    (6,5):60000}

distance3 = {(1,2):150,
    (1,3):200,
    (2,4):95,
    (2,5):100,
    (3,4):120,
    (4,5):50,
    (4,6):75,
    (5,6):60,
    (6,4):75,
    (6,5):60}

transport_rate3 = {(1,2):0.10,
    (1,3):0.15,
    (2,4):0.24,
    (2,5):0.12,
    (3,4):0.12,
    (4,5):0.25,
    (4,6):0.08,
    (5,6):0.11,
    (6,4):0.08,
    (6,5):0.11}

# Convert loss per 100km to fraction per edge
loss3 = {(1,2):0.03,
    (1,3):0.035,
    (2,4):0.032,
    (2,5):0.015,
    (3,4):0.02,
    (4,5):0.022,
    (4,6):0.018,
    (5,6):0.031,
    (6,4):0.018,
    (6,5):0.031}

max_supply3 = {1:100000,
    2:60000,
    3:25000,
    4:0,
    5:0,
    6:60000}

m3 = gp.Model("Olive_LMP_Sc3")

# Variables
Supply = m3.addVars(N, lb=0, name="Supply")
Flow = m3.addVars(E3, lb=0, name="Flow")

# Objective
supply_cost_term = gp.quicksum(
    supply_cost[n]*Supply[n] for n in N)

transport_cost_term = gp.quicksum(
    distance3[e]/100*transport_rate3[e]*Flow[e] for e in E)

m3.setObjective(supply_cost_term + transport_cost_term, GRB.MINIMIZE)

# Constraints

# Supply limit
supply_limit = m3.addConstrs(
    (Supply[n] <= max_supply3[n] for n in N), name="SupplyLimit")

# Flow capacity
capacity_con = m3.addConstrs(
    (Flow[e] <= capacity3[e] for e in E), name="Capacity")

# Node balance constraints
balance = {}

for n in N:

    inflow = gp.quicksum(
        Flow[i,j]*(1-loss3[(i,j)]*distance3[i,j]/100) for (i,j) in E if j==n)

    outflow = gp.quicksum(
        Flow[i,j] for (i,j) in E if i==n)

    balance[n] = m3.addConstr(
        Supply[n] + inflow - outflow == demand[n],
        name=f"Balance_{n}")

m3.optimize()

print("\nOptimal Objective Value")
print(m3.objVal)

print("\nOptimal Supply")
for n in N:
    print(f"Node {n}:", Supply[n].X)

print("\nOptimal Flows")
for e in E:
    print(f"{e}:", Flow[e].X)
    
#Dual Values
print("\nDual Values (Shadow Prices)")

print("\nNode Balance Duals")
for n in N:
    print(f"Node {n}:", balance[n].Pi)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 20 rows, 16 columns and 36 nonzeros
Model fingerprint: 0x14a0227a
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 1e+05]
Presolve removed 15 rows and 6 columns
Presolve time: 0.01s
Presolved: 5 rows, 10 columns, 17 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.2755232e+04   1.706402e+04   0.000000e+00      0s
       7    1.8118870e+05   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.02 seconds (0.00 work units)
Optimal objective  1.811886986e+05

Optimal Objective Value
181188.69863397835

Optimal Supply
Node 1: 85376.3440860215
Node 2: 9754.03097438156
Node 3: 25000.0
Node 4: 0.0
Node 

## Scenario 4: Increase supply in the south and transport capacity to match


In [107]:
#Scenario 4: Increase supply in the south and transport capacity to match

# Edges (from_node, to_node)
E3 = [(1,2),(1,3),(2,4),(2,5),
    (3,4),(4,5),(4,6),(5,6),
    (6,4),(6,5)]

# Edge parameters
capacity4 = {(1,2):100000,
    (1,3):100000,
    (2,4):100000,
    (2,5):100000,
    (3,4):100000,
    (4,5):100000,
    (4,6):100000,
    (5,6):100000,
    (6,4):100000,
    (6,5):100000}

distance4 = {(1,2):150,
    (1,3):200,
    (2,4):95,
    (2,5):100,
    (3,4):120,
    (4,5):50,
    (4,6):75,
    (5,6):60,
    (6,4):75,
    (6,5):60}

transport_rate4 = {(1,2):0.10,
    (1,3):0.15,
    (2,4):0.24,
    (2,5):0.12,
    (3,4):0.12,
    (4,5):0.25,
    (4,6):0.08,
    (5,6):0.11,
    (6,4):0.08,
    (6,5):0.11}

# Convert loss per 100km to fraction per edge
loss4 = {(1,2):0.03,
    (1,3):0.035,
    (2,4):0.032,
    (2,5):0.015,
    (3,4):0.02,
    (4,5):0.022,
    (4,6):0.018,
    (5,6):0.031,
    (6,4):0.018,
    (6,5):0.031}

max_supply4 = {1:100000,
    2:60000,
    3:110000,
    4:90000,
    5:90000,
    6:90000}

m4 = gp.Model("Olive_LMP_Sc4")

# Variables
Supply = m4.addVars(N, lb=0, name="Supply")
Flow = m4.addVars(E3, lb=0, name="Flow")

# Objective
supply_cost_term = gp.quicksum(
    supply_cost[n]*Supply[n] for n in N)

transport_cost_term = gp.quicksum(
    distance4[e]/100*transport_rate4[e]*Flow[e] for e in E)

m4.setObjective(supply_cost_term + transport_cost_term, GRB.MINIMIZE)

# Constraints

# Supply limit
supply_limit = m4.addConstrs(
    (Supply[n] <= max_supply4[n] for n in N), name="SupplyLimit")

# Flow capacity
capacity_con = m4.addConstrs(
    (Flow[e] <= capacity4[e] for e in E), name="Capacity")

# Node balance constraints
balance = {}

for n in N:

    inflow = gp.quicksum(
        Flow[i,j]*(1-loss4[(i,j)]*distance3[i,j]/100) for (i,j) in E if j==n)

    outflow = gp.quicksum(
        Flow[i,j] for (i,j) in E if i==n)

    balance[n] = m4.addConstr(
        Supply[n] + inflow - outflow == demand[n],
        name=f"Balance_{n}")

m4.optimize()

print("\nOptimal Objective Value")
print(m4.objVal)

print("\nOptimal Supply")
for n in N:
    print(f"Node {n}:", Supply[n].X)

print("\nOptimal Flows")
for e in E:
    print(f"{e}:", Flow[e].X)
    
#Dual Values
print("\nDual Values (Shadow Prices)")

print("\nNode Balance Duals")
for n in N:
    print(f"Node {n}:", balance[n].Pi)

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 20 rows, 16 columns and 36 nonzeros
Model fingerprint: 0xad9f69ef
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+04, 1e+05]
Presolve removed 14 rows and 2 columns
Presolve time: 0.02s
Presolved: 6 rows, 14 columns, 22 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    3.0410542e+03   2.071053e+04   0.000000e+00      0s
       9    1.7344059e+05   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.03 seconds (0.00 work units)
Optimal objective  1.734405888e+05

Optimal Objective Value
173440.58879501195

Optimal Supply
Node 1: 41884.8167539267
Node 2: 0.0
Node 3: 102422.37418261281
Node 4: 0.0
Node 5: